# Infrastructure Setup

In [1]:
pip install --no-cache-dir -r requirements.txt


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Imports & Setup

In [ ]:
import os
from enum import Enum
from typing import List, Dict, Any, Optional
from io import BytesIO
import re
import json
import requests
import arxiv
import PyPDF2
from dotenv import load_dotenv
from langchain.agents.agent import AgentExecutor, AgentAction
from langchain.agents import create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import Tool
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

#### Load env variables & assign it to variables

In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# CHROMA_API_KEY = os.getenv("CHROMA_API_KEY")

In [3]:
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")
print("Groq API key loaded successfully!")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file")
print("OpenAI API key loaded successfully!")

Groq API key loaded successfully!
OpenAI API key loaded successfully!


# Create & define tools and other helper functions that our agents can use

In [4]:
# Simple ArXiv Search Tool
def arxiv_search(query, max_results=10):
    """Search for papers on arXiv and return basic info.
    """
    arxiv_client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=max_results)
    results = []
    for paper in arxiv_client.results(search):
        results.append({
            "id": paper.entry_id,
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            "url": paper.pdf_url,
            "abstract": paper.summary,
            "published": paper.published
        })
    return results

In [5]:
# Download ArXiv paper as pdf and parse text
def get_paper_and_parse(paper_id, max_char=10000):
    """Download a paper from arXiv by ID and extract text."""
    search = arxiv.Search(id_list=[paper_id])
    client = arxiv.Client()
    paper = next(client.results(search))
    response = requests.get(paper.pdf_url)
    pdf_file = BytesIO(response.content)
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()

    return text[:max_char]

In [6]:
# Text Summarizer for the paper's contents
def paper_summarizer(text: str, 
                     groq_api_key: str=GROQ_API_KEY, 
                     max_char: int=30000):
    """Summarize text using Groq."""
    model = ChatGroq(api_key=groq_api_key, 
                     model_name="meta-llama/llama-4-maverick-17b-128e-instruct")
    prompt = f"Summarize this academic paper: {text[:max_char]}"
    return model.invoke(prompt).content

In [7]:
# Wrapper Function to create all tools above
def create_tools(groq_api_key: str,
                 max_search_results: int = 10,
                 max_char_download: int = 10000,
                 max_char_summarize: int=30000):
    """Create the tools for our agent."""

    tools = [
        Tool(
            name="search_arxiv",
            description="Search for academic papers on arXiv. Input is a search query.",
            func=lambda q: arxiv_search(q, max_search_results)
        ),
        Tool(
            name="download_paper",
            description="Download a paper from arXiv and extract its text. Input is a paper ID.",
            func=lambda id: get_paper_and_parse(id, max_char_download)
        ),
        Tool(
            name="summarize_text",
            description="Summarize a piece of text. Input is the text to summarize.",
            func=lambda t: paper_summarizer(t, groq_api_key, max_char_summarize)
        )
    ]

    return tools

# Agent Designs

Let's define the system prompt which we will use across all our implementations of this research agent

In [8]:
system_prompt = """
    You are a helpful AI assistant who is an expert research assistant.
    Your objective is to assist the user with comprehensive literature review by following the below steps as deemed appropriate:
    1. Take the user query 
    2. Search ArXiv for highly relevant papers to the query
    3. Fetch the metadata & contents of each paper from the search results
    4. Use the retrieved information to answer the user query by summarizing the contents of the paper into a dense block of relevant insights. You are expected to provide citations explicitly
    """

#### 1. Vanilla LLM Augmented

In [9]:
class VanillaLLMAugmented():
    def __init__(self, system_prompt, groq_api_key, openai_api_key, model="gpt-4o"):
        # Initialize various components of this agent
        self.llm = ChatOpenAI(model=model,
                              api_key=openai_api_key
                              )
        
        self.tools = create_tools(groq_api_key=groq_api_key)
        
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])
        
        self.agent = create_openai_tools_agent(llm=self.llm, 
                                               tools=self.tools,
                                               prompt=self.prompt)
        
        self.agent_executor = AgentExecutor(agent=self.agent, 
                                            tools=self.tools, 
                                            verbose=True)
        
    def run(self, query):
        return self.agent_executor.invoke({"input": query})

In [11]:
# Test the agent with a couple of sample queries

vanilla_llm_agent = VanillaLLMAugmented(system_prompt=system_prompt,
                                        groq_api_key=GROQ_API_KEY,
                                        openai_api_key=OPENAI_API_KEY,
                                        model="gpt-4o-mini")

queries = [
    "What are the breakthroughs in the field of AI in the last decade?",
    "What are the latest developments in quantum computing?",
    "How has protein structure discovery changed over the years?"
    ]

responses = {query: vanilla_llm_agent.run(query)["output"] for query in queries}



> Entering new AgentExecutor chain...

Invoking: `search_arxiv` with `breakthroughs in artificial intelligence in the last decade`


[{'id': 'http://arxiv.org/abs/2106.13415v1', 'title': 'Building Intelligent Autonomous Navigation Agents', 'authors': ['Devendra Singh Chaplot'], 'url': 'http://arxiv.org/pdf/2106.13415v1', 'abstract': "Breakthroughs in machine learning in the last decade have led to `digital\nintelligence', i.e. machine learning models capable of learning from vast\namounts of labeled data to perform several digital tasks such as speech\nrecognition, face recognition, machine translation and so on. The goal of this\nthesis is to make progress towards designing algorithms capable of `physical\nintelligence', i.e. building intelligent autonomous navigation agents capable\nof learning to perform complex navigation tasks in the physical world involving\nvisual perception, natural language understanding, reasoning, planning, and\nsequential decision making. Despite several 

In [14]:
print(queries[0], "\n")
print(responses[queries[0]])

What are the breakthroughs in the field of AI in the last decade? 

Over the past decade, significant breakthroughs in artificial intelligence (AI) have transformed various fields, as evidenced by a recent literature review of relevant research. Here’s a summary of the key insights gathered from the comprehensive analysis of AI advancements:

1. **Intelligent Autonomous Navigation**: Research by Chaplot (2021) focuses on developing intelligent navigation agents through a combination of digital and physical intelligence. The work emphasizes end-to-end reinforcement learning and modular algorithms, targeting improvements in localization, mapping, and long-term planning.

2. **Quantum AI in Climate Change**: Singh et al. (2021) argue that Quantum Artificial Intelligence (QAI) may significantly improve climate modeling. By enhancing data processing capabilities, QAI can play a critical role in climate prediction and adaptation strategies amid growing climate-related challenges.

3. **AI Au

#### Prompt chaining

A chain of prompts mean that the output of one LLM call is fed into another.
    

In [53]:
def setup_prompt_chain(groq_api_key, openai_api_key):
    search_llm = ChatOpenAI(api_key=openai_api_key, 
                            model="gpt-4o-mini")
    synthesis_llm = ChatGroq(api_key=groq_api_key, 
                             model_name="meta-llama/llama-4-maverick-17b-128e-instruct")

    tools = create_tools(groq_api_key=groq_api_key)

    # Search agent prompt
    search_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a research assistant that helps find relevant papers on arXiv.
Your job is to convert a user's research question into the best possible search query for arXiv.
Focus on extracting key technical terms and concepts.
Respond ONLY with the search query, nothing else."""),
        ("human", "{input}")
    ])

    # Paper selection agent prompt
    selection_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a research assistant that helps select the most relevant paper from search results.
Given a list of papers and the original research question, select the SINGLE most relevant paper ID.
Respond ONLY with the paper ID, not the entire URL, nothing else."""),
        ("human", "Research question: {original_query}\nSearch results: {search_results}")
    ])

    # Analysis agent prompt
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a research expert that analyzes academic papers.
Given the text of a paper, identify the key methodologies, findings, and implications.
Focus on how this paper addresses the user's original question."""),
        ("human", "Original question: {original_query}\nPaper text: {paper_text}")
    ])

    # Final answer prompt
    synthesis_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a helpful AI research assistant that synthesizes information into clear,
concise responses. Create a comprehensive yet accessible answer to the user's question based on the
technical analysis provided."""),
        ("human", "Original question: {original_query}\nTechnical analysis: {analysis}")
    ])

    return {
        "search_llm": search_llm,
        "synthesis_llm": synthesis_llm,
        "tools": tools,
        "prompts": {
            "search": search_prompt,
            "selection": selection_prompt,
            "analysis": analysis_prompt,
            "synthesis": synthesis_prompt
        }
    }

In [54]:
def run_prompt_chain(query, groq_api_key, openai_api_key):
    # Setup the chain components
    chain = setup_prompt_chain(groq_api_key, openai_api_key)
    search_llm = chain["search_llm"]
    synthesis_llm = chain["synthesis_llm"]
    prompts = chain["prompts"]

    print("📝 STARTING PROMPT CHAIN")
    print(f"📋 Original query: {query}")

    # STEP 1: Convert question to search query
    print("\n🔍 Step 1: Converting question to optimal search query...")
    search_query = search_llm.invoke(prompts["search"].format(input=query)).content
    print(f"🔍 Generated search query: {search_query}")
    search_query = search_query.replace('"', '').replace("'", "").strip()
    
    # STEP 2: Search for papers
    print("\n📚 Step 2: Searching for papers on arXiv...")
    search_results = arxiv_search(search_query)
    print(f"📚 Found {len(search_results)} papers")

    # STEP 3: Select the most relevant paper
    print("\n🔎 Step 3: Selecting the most relevant paper...")
    paper_id = search_llm.invoke(
        prompts["selection"].format(
            original_query=query,
            search_results=search_results
        )
    ).content
    print(f"🔎 Selected paper ID: {paper_id}")

    # STEP 4: Download the selected paper
    print("\n📄 Step 4: Downloading the selected paper...")
    paper_text = get_paper_and_parse(paper_id)
    print(f"📄 Downloaded {len(paper_text)} characters of paper text")

    # STEP 5: Analyze the paper
    print("\n🧪 Step 5: Analyzing the paper...")
    analysis = search_llm.invoke(
        prompts["analysis"].format(
            original_query=query,
            paper_text=paper_text
        )
    ).content
    print(f"🧪 Analysis complete")

    # STEP 6: Final synthesis
    print("\n🔮 Step 6: Synthesizing final response...")
    final_response = synthesis_llm.invoke(
        prompts["synthesis"].format(
            original_query=query,
            analysis=analysis
        )
    ).content

    print("\n✅ PROMPT CHAIN COMPLETE")

    return {
        "search_query": search_query,
        "selected_paper": next((p for p in search_results if p["id"] == paper_id), None),
        "analysis": analysis,
        "response": final_response
    }

In [55]:
if not GROQ_API_KEY or not OPENAI_API_KEY:
    raise ValueError("API keys not found. Make sure GROQ_API_KEY and OPENAI_API_KEY are set.")

query = "What were the major breakthroughs in AI in the last 10 years?"
result = run_prompt_chain(query, GROQ_API_KEY, OPENAI_API_KEY)

print("----- QUERY -----")
print("\n\n\n----- FINAL RESPONSE -----")
print(result["response"])

📝 STARTING PROMPT CHAIN
📋 Original query: What were the major breakthroughs in AI in the last 10 years?

🔍 Step 1: Converting question to optimal search query...
🔍 Generated search query: "AI breakthroughs last 10 years"

📚 Step 2: Searching for papers on arXiv...
📚 Found 10 papers

🔎 Step 3: Selecting the most relevant paper...
🔎 Selected paper ID: 2505.16771v1

📄 Step 4: Downloading the selected paper...
📄 Downloaded 10000 characters of paper text

🧪 Step 5: Analyzing the paper...
🧪 Analysis complete

🔮 Step 6: Synthesizing final response...

✅ PROMPT CHAIN COMPLETE
----- QUERY -----



----- FINAL RESPONSE -----
The last decade has seen significant breakthroughs in AI, transforming the field and paving the way for future advancements. The major technical milestones include:

1. **GPU-accelerated deep learning (2009)**: The adoption of Graphics Processing Units (GPUs) for deep learning revolutionized model training speed and scalability, enabling the development of more complex AI mo

#### Routing

Define the Router class

In [56]:
class Router:
    def __init__(self, openai_api_key):
        self.llm = ChatOpenAI(api_key=openai_api_key, model="gpt-4o-mini")
        self.router_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a query classifier that determines the type of research question.
Analyze the user's research question and classify it into EXACTLY ONE of the following categories 
that is most appropriate given the user query:
1. "latest_developments" - Questions about recent advancements or state-of-the-art in a field
2. "deep_dive" - Questions requiring in-depth analysis of specific techniques or methodologies
3. "comparison" - Questions comparing different approaches, methods, or theories
4. "application" - Questions about applying research to specific domains or problems

Respond ONLY with the category name in lowercase, nothing else."""),
            ("human", "{input}")
        ])
        self.valid_types = ["latest_developments", "deep_dive", "comparison", "application"]

    def route(self, query):
        response = self.llm.invoke(self.router_prompt.format(input=query))
        route = response.content.strip().lower()
        if route not in self.valid_types:
            route = "latest_developments"
        return route

Define all agents, one each for a specific route: LatestDevelopmentAgent, DeepDiveAgent, ComparisonAgent & ApplicationAgent

In [57]:
class LatestDevelopmentsAgent:
    def __init__(self, openai_api_key, groq_api_key, tools):
        self.llm = ChatOpenAI(api_key=openai_api_key, model="gpt-4o-mini")
        self.tools = tools
        self.groq_api_key = groq_api_key
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a research assistant specializing in the latest developments and
state-of-the-art research. Your goal is to find the most recent papers on the relevant topics 
inferred from the user query below and summarize the latest developments ONLY in that topic."""),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])

    def process(self, query):
        agent = create_openai_tools_agent(self.llm, self.tools, self.prompt)
        agent_executor = AgentExecutor(agent=agent, tools=self.tools, verbose=True)
        result = agent_executor.invoke({"input": query})
        return result["output"]

In [58]:
class DeepDiveAgent:
    def __init__(self, openai_api_key, groq_api_key, tools):
        self.llm = ChatOpenAI(api_key=openai_api_key, model="o3-mini")
        self.tools = tools
        self.groq_api_key = groq_api_key
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a research assistant specializing in deep technical analysis.
Your goal is to find and thoroughly analyze papers on specific techniques or methodologies 
pertaining to the topic. Focus on understanding the core principles, granular details, 
and theoretical foundations."""),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])

    def process(self, query):
        agent = create_openai_tools_agent(self.llm, self.tools, self.prompt)
        agent_executor = AgentExecutor(agent=agent, tools=self.tools, verbose=True)
        result = agent_executor.invoke({"input": query})
        return result["output"]

In [59]:
class ComparisonAgent:
    def __init__(self, openai_api_key, groq_api_key, tools):
        self.llm = ChatOpenAI(api_key=openai_api_key, model="o3-mini")
        self.tools = tools
        self.groq_api_key = groq_api_key
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a research assistant specializing in comparative analysis.
Your goal is to find papers that compare different approaches, methods, or theories and
analyze the strengths and weaknesses of each. Create a balanced assessment that highlights
the key differences."""),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])

    def process(self, query):
        agent = create_openai_tools_agent(self.llm, self.tools, self.prompt)
        agent_executor = AgentExecutor(agent=agent, tools=self.tools, verbose=True)
        result = agent_executor.invoke({"input": query})
        return result["output"]

In [60]:
class ApplicationAgent:
    def __init__(self, openai_api_key, groq_api_key, tools):
        self.llm = ChatOpenAI(api_key=openai_api_key, model="o3-mini")
        self.tools = tools
        self.groq_api_key = groq_api_key
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a research assistant specializing in practical applications.
Your goal is to find papers that demonstrate how research is applied to real-world problems
and domains. Focus on implementation details, case studies, and practical outcomes."""),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])

    def process(self, query):
        agent = create_openai_tools_agent(self.llm, self.tools, self.prompt)
        agent_executor = AgentExecutor(agent=agent, tools=self.tools, verbose=True)
        result = agent_executor.invoke({"input": query})
        return result["output"]

In [61]:
def setup_routing(openai_api_key, groq_api_key):
    tools = create_tools(groq_api_key)
    router = Router(openai_api_key)
    agents = {
        "latest_developments": LatestDevelopmentsAgent(openai_api_key, groq_api_key, tools),
        "deep_dive": DeepDiveAgent(openai_api_key, groq_api_key, tools),
        "comparison": ComparisonAgent(openai_api_key, groq_api_key, tools),
        "application": ApplicationAgent(openai_api_key, groq_api_key, tools)
    }
    return router, agents


In [62]:
def run_routing_system(query, openai_api_key, groq_api_key):
    router, agents = setup_routing(openai_api_key, groq_api_key)

    print("🚦 ROUTING SYSTEM INITIATED")
    print(f"📋 Query: {query}")

    # Step 1: Route the query
    print("\n🔀 Routing query...")
    query_type = router.route(query)
    print(f"🔀 Query classified as: {query_type}")

    # Step 2: Process with the appropriate agent
    print(f"\n🔬 Processing with specialized {query_type} agent...")
    response = agents[query_type].process(query)

    print("\n✅ ROUTING COMPLETE")

    return {
        "query_type": query_type,
        "response": response
    }

In [63]:
if not GROQ_API_KEY or not OPENAI_API_KEY:
    raise ValueError("API keys not found. Make sure GROQ_API_KEY and OPENAI_API_KEY are set.")

sample_queries = {
    "latest_developments": "What are the latest advancements in quantum computing?",
    "deep_dive": "How do transformers handle attention mechanisms in deep learning?",
    "comparison": "Compare LSTM and transformer approaches for sequence modeling",
    "application": "How is reinforcement learning applied to robotics?"
}

query = sample_queries["deep_dive"]

result = run_routing_system(query, OPENAI_API_KEY, GROQ_API_KEY)

print("\n----- FINAL RESPONSE -----")
print(result["response"])
print("\n--------------------------")

🚦 ROUTING SYSTEM INITIATED
📋 Query: How do transformers handle attention mechanisms in deep learning?

🔀 Routing query...
🔀 Query classified as: deep_dive

🔬 Processing with specialized deep_dive agent...


> Entering new AgentExecutor chain...
Transformers use attention mechanisms—specifically the scaled dot-product self-attention—to allow the model to weigh different parts of the input sequence dynamically, rather than relying solely on sequential processing. Here’s a detailed breakdown of how this works:

1. Input Representation:  
   Each token in the input is first converted into a vector using an embedding. Positional encodings are then added to these embeddings to inject information about the position of each token, since the attention mechanism itself is permutation-invariant.

2. Query, Key, and Value Vectors:  
   For every input token, three different vectors are computed through learned linear transformations:
   • Query (Q): Represents the token's “search” for relevant con

#### Fully autonomous agent

Imports

Define configs using Enums

In [67]:
class AgentAction(Enum):
    USE_TOOL = "use_tool"
    ROUTE = "route"
    FINAL_ANSWER = "final_answer"

class ResearchPath(Enum):
    OVERVIEW = "overview"
    DEEP_DIVE = "deep_dive"
    COMPARISON = "comparison"
    TECHNICAL = "technical"

Define schemas for LLM responses using Pydantic

In [68]:
class ToolActionInput(BaseModel):
    tool_name: str = Field(description="Name of the tool to use")
    tool_input: str = Field(description="Input to provide to the tool")

class RouteActionInput(BaseModel):
    path: str = Field(description="Research path to take")

class FinalAnswerInput(BaseModel):
    answer: str = Field(description="Final answer to the query")

class AgentDecision(BaseModel):
    thought: str = Field(description="Detailed reasoning about the current state and what to do next")
    action: str = Field(description="Action to take: use_tool, route, or final_answer")
    action_input: Union[ToolActionInput, RouteActionInput, FinalAnswerInput] = Field(
        description="Input for the action"
    )

Define the state container that will be used by the agent through different paths and will be used to manage state & the agent's resoning process

In [69]:
class AgentState:
    def __init__(self, query: str):
        self.query = query
        self.current_path: Optional[ResearchPath] = None
        self.tool_results: List[Dict[str, Any]] = []
        self.working_memory: List[str] = []
        self.final_answer: Optional[str] = None
        self.conversation_history: List[Dict[str, Any]] = []

    def add_tool_result(self, tool_name: str, tool_input: str, tool_output: Any) -> None:
        self.tool_results.append({
            "tool_name": tool_name,
            "tool_input": tool_input,
            "tool_output": tool_output
        })

    def add_to_memory(self, thought: str) -> None:
        self.working_memory.append(thought)

    def set_path(self, path: ResearchPath) -> None:
        self.current_path = path

    def set_final_answer(self, answer: str) -> None:
        self.final_answer = answer

    def add_message(self, role: str, content: str) -> None:
        self.conversation_history.append({
            "role": role,
            "content": content
        })

    def to_dict(self) -> Dict[str, Any]:
        return {
            "query": self.query,
            "current_path": self.current_path.value if self.current_path else None,
            "tool_results": self.tool_results,
            "working_memory": self.working_memory,
            "final_answer": self.final_answer,
        }

Finally, define the Agent itself

In [70]:
class ResearchAgent:
    def __init__(self, openai_api_key: str, groq_api_key: str):
        self.llm = ChatOpenAI(api_key=openai_api_key, model="o3-mini")
        self.groq_api_key = groq_api_key
        self.tools = create_tools(groq_api_key)
        self.parser = JsonOutputParser(pydantic_object=AgentDecision)
        self.decision_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a research assistant agent capable of dynamically making decisions 
about how to answer research questions. You must explicitly reason through THREE types of decisions:

1. ROUTING - Determine which research path would best address the query:
   - OVERVIEW: For broad, introductory questions needing a general summary
   - DEEP_DIVE: For questions requiring in-depth focus on a specific topic
   - COMPARISON: For questions comparing multiple approaches or methods
   - TECHNICAL: For questions requiring detailed technical explanations

2. TOOL SELECTION - Determine which tool to use next from:
   - arxiv_search: Search for academic papers (input: search query)
   - get_paper_and_parse: Get the text of a specific paper (input: paper ID)
   - paper_summarizer: Summarize a piece of text (input: text to summarize)

3. FINAL ANSWER - Determine if you have enough information to provide a final answer

For each decision, you MUST use the following process:
1. Think through the options step-by-step
2. Consider the query and current state
3. Make an explicit choice
4. Explain your reasoning

{format_instructions}"""),
            
            ("human", """
QUERY: {query}
CURRENT PATH: {current_path}
TOOL RESULTS:
{tool_results}
WORKING MEMORY:
{working_memory}

Based on the above, what should I do next? Think through whether I should choose a research path, use a tool, or provide a final answer.
""")
        ])

    def _format_tool_results(self, tool_results: List[Dict[str, Any]]) -> str:
        if not tool_results:
            return "No tools have been used yet."

        result_strings = []
        for i, result in enumerate(tool_results):
            result_str = f"Tool {i+1}: {result['tool_name']}\n"
            result_str += f"Input: {result['tool_input']}\n"

            if result['tool_name'] == "search_arxiv":
                papers = result['tool_output']
                result_str += "Output (Papers):\n"
                for j, paper in enumerate(papers):
                    result_str += f"  Paper {j+1}: {paper['title']}\n"
                    result_str += f"  ID: {paper['id']}\n"
                    result_str += f"  Authors: {', '.join(paper['authors'])}\n"
                    result_str += f"  URL: {paper['url']}\n\n"
            elif result['tool_name'] == "download_paper":
                paper_text = result['tool_output']
                if len(paper_text) > 500:
                    paper_text = paper_text[:500] + "... [text truncated]"
                result_str += f"Output (Paper Text):\n{paper_text}\n\n"
            elif result['tool_name'] == "summarize_text":
                result_str += f"Output (Summary):\n{result['tool_output']}\n\n"

            result_strings.append(result_str)

        return "\n".join(result_strings)

    def _format_working_memory(self, working_memory: List[str]) -> str:
        if not working_memory:
            return "No thoughts recorded yet."

        memory_strings = []
        for i, thought in enumerate(working_memory):
            memory_strings.append(f"Thought {i+1}: {thought}")

        return "\n".join(memory_strings)

    def _get_next_action(self, state: AgentState) -> Dict[str, Any]:
      prompt_args = {
          "query": state.query,
          "current_path": state.current_path.value if state.current_path else "Not selected yet",
          "tool_results": self._format_tool_results(state.tool_results),
          "working_memory": self._format_working_memory(state.working_memory),
          "format_instructions": self.parser.get_format_instructions()
      }
      
      response = self.llm.invoke(self.decision_prompt.format(**prompt_args))
      
      try:
          response_text = response.content
          match = re.search(r'(\{.*\})', response_text, re.DOTALL)
          if match:
              json_str = match.group(1)
              action_data = json.loads(json_str)
              return {
                  "thought": action_data.get("thought", "No reasoning provided"),
                  "action": action_data.get("action", "final_answer"),
                  "action_input": action_data.get("action_input", {"answer": "I don't have enough information to answer."})
              }
          else:
              return {
                  "thought": "Failed to extract JSON from response",
                  "action": "final_answer",
                  "action_input": {
                      "answer": "I encountered an error in my reasoning process."
                  }
              }
              
      except Exception as e:
          print(f"Error parsing LLM response: {e}")
          print(f"Problem response: {response_text}")
          return {
              "thought": f"There was an error parsing my previous response: {str(e)}",
              "action": "final_answer",
              "action_input": {
                  "answer": "I encountered a technical error in my reasoning process. Let me provide a direct answer based on my knowledge: Transformers and LSTMs differ primarily in how they process sequential data - LSTMs process data sequentially while transformers can process the entire sequence at once through self-attention mechanisms."
              }
          }

    def _execute_tool(self, tool_name: str, tool_input: str) -> Any:
        tool = next((t for t in self.tools if t.name == tool_name), None)
        if not tool:
            return f"Error: Tool '{tool_name}' not found."
        try:
            result = tool.func(tool_input)
            return result
        except Exception as e:
            return f"Error executing tool: {str(e)}"

    def _route_query(self, path_name: str) -> ResearchPath:
        try:
            return ResearchPath(path_name)
        except ValueError:
            return ResearchPath.OVERVIEW

    def process_query(self, query: str, max_steps: int = 10) -> Dict[str, Any]:
        state = AgentState(query)
        print("🤖 RESEARCH AGENT INITIATED")
        print(f"📋 Query: {query}")
        for step in range(max_steps):
            print(f"\n🔄 Step {step+1}: Reasoning...")
            action_data = self._get_next_action(state)
            thought = action_data.get("thought", "No reasoning provided")
            state.add_to_memory(thought)
            print(f"💭 Thought: {thought}")
            action = action_data.get("action")
            action_input = action_data.get("action_input", {})
            if action == AgentAction.USE_TOOL.value:
                tool_name = action_input.get("tool_name")
                tool_input = action_input.get("tool_input")
                print(f"🔧 Using Tool: {tool_name}")
                print(f"📥 Tool Input: {tool_input}")
                tool_output = self._execute_tool(tool_name, tool_input)
                state.add_tool_result(tool_name, tool_input, tool_output)
                print(f"📤 Tool Output: {type(tool_output)} with {len(str(tool_output))} chars")
            elif action == AgentAction.ROUTE.value:
                path_name = action_input.get("path")
                path = self._route_query(path_name)
                state.set_path(path)
                print(f"🔀 Routing to Path: {path.value}")
            elif action == AgentAction.FINAL_ANSWER.value:
                answer = action_input.get("answer")
                state.set_final_answer(answer)
                print(f"✅ Final Answer Ready")
                break
            else:
                print(f"⚠️ Unknown Action: {action}")
                state.set_final_answer(f"I encountered an error in my reasoning process. Unknown action: {action}")
                break
        else:
            state.set_final_answer("I've reached the maximum number of reasoning steps without finding a complete answer. Here's what I've learned so far: " +
                                 "\n\n".join(state.working_memory[-3:]))
            print("⚠️ Maximum steps reached without final answer")
        print("\n✨ REASONING COMPLETE")
        return {
            "final_answer": state.final_answer,
            "state": state.to_dict()
        }

In [72]:
if not GROQ_API_KEY or not OPENAI_API_KEY:
    raise ValueError("API keys not found. Make sure GROQ_API_KEY and OPENAI_API_KEY are set.")

agent = ResearchAgent(OPENAI_API_KEY, GROQ_API_KEY)

query = """Break down the Reinforcement Learning area into different paradigms, 
discuss types of algorithms, problems solved, techniques etc.
Give a high level overview of the research breakthroughs in that area and how it's commercially used"""

result = agent.process_query(query)

print("\n\n\n----- FINAL ANSWER -----")
print(result["final_answer"])

🤖 RESEARCH AGENT INITIATED
📋 Query: Break down the Reinforcement Learning area into different paradigms, 
discuss types of algorithms, problems solved, techniques etc.
Give a high level overview of the research breakthroughs in that area and how it's commercially used

🔄 Step 1: Reasoning...
💭 Thought: The query asks for a broad overview on Reinforcement Learning, including a breakdown into paradigms, discussion of algorithms, problems solved, techniques, research breakthroughs, and commercial applications. This clearly indicates that an OVERVIEW approach is best. The next step should be to select the OVERVIEW research path.
🔀 Routing to Path: overview

🔄 Step 2: Reasoning...
💭 Thought: The query asks for a comprehensive and broad overview of Reinforcement Learning, breaking down its paradigms, algorithms, solved problems, techniques, research breakthroughs, and commercial applications. This aligns perfectly with an OVERVIEW research path. Therefore, I will use the research path decisi